## Open Government Data, provided by **OpenDataZurich**
*Autogenerated Python starter code for data set with identifier* **prd_ssz_ogd_katalog_downloads**

## Dataset
# **Download-Zahlen des Open-Data-Katalogs der Stadt Zürich**

## Description

Dieser Datensatz enthält aggregierte Download-Zahlen für Ressourcen des Open-Data-Katalogs der Stadt Zürich ([data.stadt-zuerich.ch](https://data.stadt-zuerich.ch/)). Die Daten werden täglich aus den NGINX-Logs der Plattform extrahiert und enthalten Informationen zu Anzahl der Downloads (hit_count), eindeutigen IPs (distinct_ips), verwendeten User-Agents, Ressourcen-IDs, Datensatz-IDs sowie den zugehörigen URLs.

**Umfang der erfassten Downloads und Filterlogik**

Die Daten erfassen Downloads über den Webbrowser (über die Frontend-Oberfläche) sowie direkte Downloads und API-Aufrufe (z. B. über datastore_search_sql). Interne Anfragen, insbesondere solche, die vom Frontend-Portal des Open-Data-Katalogs selbst ausgehen (z. B. zur Anzeige von Ressourcen in der UI), sind vollständig herausgefiltert. Anfragen von externen Crawlern (z. B. YandexBot, Bytespider) sind nicht herausgefiltert, da sie als Teil der tatsächlichen Nutzung der Daten angesehen werden und für die Analyse relevant sind.

**Ergänzung von Metadaten**

Da die ursprünglichen Logs nicht immer vollständige Informationen zu Ressourcen- und Datensatznamen enthalten, werden diese über die CKAN-API ergänzt, um die Datenqualität zu sichern. Publiziert werden nur Einträge, bei denen der dataset_name sowie entweder der resource_name oder die resource_id eindeutig identifizierbar sind.

**Datenschutz und Anonymisierung**

Die Daten sind nicht personenbezogen: IP-Adressen werden nicht gespeichert, nur als Anzahl (distinct_ips) gezählt.

**Aktualisierung**

Es gibt eine Ressource je Jahr. Die Ressource des jeweils aktuellen Jahres wird täglich mit den Downloadzahlen des Vortages ergänzt.



## Data set links

[Direct link by OpenDataZurich for dataset](https://data.stadt-zuerich.ch/dataset/prd_ssz_ogd_katalog_downloads)

https://data.stadt-zuerich.ch/dataset/prd_ssz_ogd_katalog_downloads/download/ogd_katalog_downloads_2026.parquet<br>


## Metadata
- **Publisher** `Statistik Stadt Zürich, Präsidialdepartement`
- **Maintainer** `Open Data Zürich`
- **Maintainer_email** `opendata@zuerich.ch`
- **Keywords** `Ver­walt­ung`
- **Tags** `['analytics', 'download', 'github', 'nutzung', 'statistik', 'tracking']`
- **Metadata_created** `2025-05-07T14:14:37.511860`
- **Metadata_modified** `2026-10-25T08:05:57.541764`


## Imports and helper functions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Identifies requests from this notebook in the OpenDataZurich access logs.
USER_AGENT = "OpenDataZurich-StarterCode-Notebook/1.0 (lang=python; +https://github.com/opendatazurich/starter-code)"
STORAGE_OPTIONS = {"User-Agent": USER_AGENT}

In [ ]:
# helper function for reading datasets with proper separator
def get_dataset(url):
    """
    Return pandas df if url is a parquet or csv file. Return None if not.
    """
    extension = url.rsplit(".",1)[-1]
    if extension == 'parquet':
        data = pd.read_parquet(url, storage_options=STORAGE_OPTIONS)
    elif extension == 'csv':
        data = pd.read_csv(url, sep=",", on_bad_lines='warn', encoding_errors='ignore', low_memory=False, storage_options=STORAGE_OPTIONS)
        # if dataframe only has one column or fewer, the data is not comma separated; use ";" instead
        if data.shape[1] <= 1:
            data = pd.read_csv(url, sep=';', on_bad_lines='warn', encoding_errors='ignore', low_memory=False, storage_options=STORAGE_OPTIONS)
            if data.shape[1] <= 1:
                print("The data wasn't imported properly. Very likely the correct separator couldn't be found.\nPlease check the dataset manually and adjust the code.")
    else:
        print("Cannot load data! Please provide a url with csv or parquet extension.")
        data = None
    return data

## Load the data

In [ ]:
df = get_dataset('https://data.stadt-zuerich.ch/dataset/prd_ssz_ogd_katalog_downloads/download/ogd_katalog_downloads_2026.parquet')

## Analyze the data

In [ ]:
# drop columns that have no values
df.dropna(how='all', axis=1, inplace=True)

In [ ]:
print(f'The dataset has {df.shape[0]:,.0f} rows (observations) and {df.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {df.duplicated().sum()} exact duplicates in the data.')

In [ ]:
df.info(memory_usage='deep', verbose=True)

In [ ]:
df.head()

In [ ]:
# display a small random sample transposed in order to see all variables
df.sample(3).T

In [ ]:
# describe non-numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(exclude='number'))
except:
    print("No categorical data in dataset.")

In [ ]:
# describe numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(include='number'))
except:
    print("No numerical data in dataset.")

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    ax.barh(missing["column"], missing["missing_pct"], color="steelblue")
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# plot a histogram for each numerical feature
try:
    df.hist(bins=25, rwidth=.9)
    plt.tight_layout()
    plt.show()
except:
    print("No numerical data to plot.") 

In [ ]:
# continue your code here...

**Contact**: opendata@zuerich.ch